# 02 - 블록 확산과 접두사 anchor

**학습 목표**: 긴 문자열을 작은 블록으로 나누고, 완성된 접두사를 고정한 채 현재 블록만 병렬 복원합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 기능만 사용합니다.

실제 neural denoiser 대신 정답과 toy confidence를 사용합니다.

In [ ]:
MASK = '[M]'
# 접두사와 현재 canvas를 list로 분리해 이미 확정한 block이 바뀌지 않음을 명시합니다.

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[-1] + 1, prev[j] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def block_diffuse(text, block_size=8):
    truth = list(text)
    committed = []
    forwards = 0
    trace = []
    for start in range(0, len(truth), block_size):
        block = truth[start:start + block_size]
        canvas = [MASK] * len(block)
        # 보수적 threshold에서 시작해 현재 블록이 끝날 때까지 낮춥니다.
        for threshold in (0.99, 0.96, 0.92, 0.0):
            forwards += 1
            for i in range(len(block)):
                confidence = 0.995 - 0.025 * (i % 4)
                if canvas[i] == MASK and confidence >= threshold:
                    canvas[i] = block[i]
            trace.append((''.join(committed), start, threshold, canvas.copy()))
            if MASK not in canvas:
                break
        committed.extend(canvas)  # 이후 블록이 사용하는 immutable prefix
    return ''.join(committed), forwards, trace

truth = '<p>block diffusion keeps order</p>'
prediction, forwards, trace = block_diffuse(truth, block_size=8)
ned = edit_distance(prediction, truth) / max(len(prediction), len(truth))
print('prediction:', prediction)
print('forward calls:', forwards, 'vs AR calls:', len(truth))
print('toy steps/token:', round(forwards / len(truth), 3), 'NED:', ned)
print('첫 블록 trace:', trace[:4])
assert prediction == truth and ned == 0.0


## 해석

접두사는 다음 블록 동안 바뀌지 않으므로 KV cache가 가능합니다. 이 toy는 정답을 직접 참조하므로 모델 정확도를 증명하지 않습니다. 순차 깊이가 블록 수×denoising step으로 줄어드는 구조만 확인합니다.